In [1]:
import pandas as pd
import re

# Baca file CSV
df = pd.read_csv("emoji_to_text.csv")

def clean_for_token(text):
    # Ubah spasi menjadi underscore
    text = text.replace(" ", "_")
    # Ubah ke huruf kapital semua
    text = text.upper()
    # Hapus karakter selain huruf, angka, dan underscore
    text = re.sub(r"[^A-Z0-9_]", "", text)
    return text

# Terapkan ke kolom 'makna'
df["makna"] = df["makna"].apply(clean_for_token)

# Simpan ke file baru
df.to_csv("emoji_to_text_token_safe.csv", index=False)

print("Proses selesai! File tersimpan sebagai emoji_to_text_token_safe.csv")


Proses selesai! File tersimpan sebagai emoji_to_text_token_safe.csv


## Ubah emoji ke teks

In [2]:
import pandas as pd
import re

# === 1. Baca mapping emoji → token ===
mapping_df = pd.read_csv("emoji_to_text_token_safe.csv")
emoji_map = dict(zip(mapping_df["emoji"], mapping_df["makna"]))

# === 2. Buat regex pattern yang cocok untuk semua emoji ===
emoji_pattern = re.compile("|".join(re.escape(e) for e in emoji_map.keys()))

# === 3. Fungsi untuk ganti emoji ke token ===
def replace_emoji_with_token(text):
    if not isinstance(text, str):
        return text
    # Ganti setiap emoji dengan [TOKEN] dan beri spasi agar aman di tokenisasi
    text = emoji_pattern.sub(lambda m: f" [{emoji_map[m.group()]}] ", text)
    # Bersihkan spasi ganda akibat penambahan token
    text = re.sub(r"\s+", " ", text).strip()
    return text

# === 4. Baca dataset komentar ===
df = pd.read_csv("gabungan_komentar(fix2).csv")  # ganti dengan nama file kamu

# === 5. Terapkan fungsi ke kolom komentar ===
df["Komentar"] = df["Komentar"].apply(replace_emoji_with_token)

# === 6. Simpan hasil ===
df.to_csv("dataset_komentar_emoji_token.csv", index=False, encoding="utf-8-sig")

print("✅ Proses selesai! File tersimpan sebagai dataset_komentar_emoji_token.csv")


✅ Proses selesai! File tersimpan sebagai dataset_komentar_emoji_token.csv


In [3]:
import pandas as pd
import re

# === 1. Load mapping emoji → token ===
mapping_df = pd.read_csv("emoji_to_text_token_safe.csv")
emoji_map = dict(zip(mapping_df["emoji"], mapping_df["makna"]))

# === 2. Daftar skin tone modifier (Unicode U+1F3FB–U+1F3FF) ===
skin_tone_modifiers = re.compile(r'[\U0001F3FB-\U0001F3FF]')

# === 3. Buat pattern emoji berdasarkan mapping ===
emoji_pattern = re.compile("|".join(re.escape(e) for e in emoji_map.keys()))

# List untuk simpan emoji yang tidak dikenal
unknown_emojis = set()

# === 4. Fungsi ganti emoji ===
def replace_emoji_with_token(text):
    if not isinstance(text, str):
        return text
    # Hapus skin tone modifier
    clean_text = skin_tone_modifiers.sub('', text)
    
    # Cari semua emoji yang tidak ada di mapping
    for char in clean_text:
        if re.match(r'[\U0001F300-\U0001FAFF]', char):  # rentang emoji umum
            if char not in emoji_map:
                unknown_emojis.add(char)
    
    # Ganti emoji yang dikenal
    clean_text = emoji_pattern.sub(lambda m: f" [{emoji_map[m.group()]}] ", clean_text)
    
    # Bersihkan spasi ganda
    clean_text = re.sub(r"\s+", " ", clean_text).strip()
    
    return clean_text

# === 5. Baca dataset komentar ===
df = pd.read_csv("gabungan_komentar(fix2).csv")  # ganti nama file sesuai dataset kamu

# === 6. Terapkan fungsi ===
df["Komentar"] = df["Komentar"].apply(replace_emoji_with_token)

# === 7. Simpan hasil ===
df.to_csv("dataset_komentar_emoji_token(1).csv", index=False, encoding="utf-8-sig")

# === 8. Simpan daftar emoji yang belum ada di mapping ===
if unknown_emojis:
    with open("emoji_unknown.txt", "w", encoding="utf-8") as f:
        for e in sorted(unknown_emojis):
            f.write(f"{e}\n")

print("✅ Proses selesai!")
print(f"Emoji tidak dikenal disimpan di emoji_unknown.txt ({len(unknown_emojis)} ditemukan).")


✅ Proses selesai!
Emoji tidak dikenal disimpan di emoji_unknown.txt (113 ditemukan).


In [9]:
!pip uninstall googletrans
!pip install googletrans==4.0.0-rc1

^C
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.3 MB ? eta -:--:--
   ---------------- ----------------------- 0.5/1.3 MB 1.3 MB/s eta 0:00:01
   ------------------------ --------------- 0.8/1.3 MB 1.1 MB/s eta 0:00:01
   -------------------------------- ------- 1.0/1.3 MB 1.1 MB/s eta 0:00:01
   -------------------------------- ------- 1.0/1.3 MB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 1.0 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17458 sha256=6c536151a17dd8de512e740690086bbda99d688861a196c24f702c8f70065165
  Stored in directory: c:\users\asus\appdata\local\pip\cache\wheels\95\0f\04\b17a72024b56a60e499ce1a6313d283ed5ba332407155bee03
Successfully buil

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.0 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import re
import unicodedata
from googletrans import Translator

# === Konfigurasi file ===
dataset_file = "gabungan_komentar(fix2).csv"  # dataset mentah
mapping_file = "emoji_to_text_token_safe.csv"  # mapping awal
output_dataset_file = "dataset_komentar_emoji_token(1.1).csv"
output_mapping_file = "emoji_to_text_token_safe_updated(1.1).csv"
output_unknown_file = "emoji_unknown(1.1).txt"

# === 1. Load mapping awal ===
df_map = pd.read_csv(mapping_file)
emoji_map = dict(zip(df_map["emoji"], df_map["makna"]))

# === 2. Skin tone modifiers ===
skin_tone_modifiers = re.compile(r'[\U0001F3FB-\U0001F3FF]')

# === 3. Translator Google ===
translator = Translator()

# === 4. Deteksi emoji di teks ===
def find_emojis(text):
    return [ch for ch in text if re.match(r'[\U0001F300-\U0001FAFF]', ch)]

# === 5. Proses dataset ===
df = pd.read_csv(dataset_file)
unknown_emojis = set()

# Cek semua emoji di dataset
for text in df["Komentar"]:
    if isinstance(text, str):
        text_clean = skin_tone_modifiers.sub("", text)
        for e in find_emojis(text_clean):
            if e not in emoji_map:
                unknown_emojis.add(e)

# === 6. Tambahkan emoji baru ke mapping ===
if unknown_emojis:
    new_rows = []
    with open(output_unknown_file, "w", encoding="utf-8") as f:
        for e in sorted(unknown_emojis):
            f.write(f"{e}\n")
            try:
                unicode_name = unicodedata.name(e).replace("_", " ")
            except ValueError:
                unicode_name = "unknown emoji"

            result = translator.translate(unicode_name, src="en", dest="id")
            translated = result.text
            token_name = translated.upper().replace(" ", "_")
            new_rows.append({"emoji": e, "makna": token_name})

    df_new = pd.DataFrame(new_rows)
    df_map = pd.concat([df_map, df_new], ignore_index=True)
    df_map = df_map.drop_duplicates(subset=["emoji"], keep="first")
    emoji_map.update(dict(zip(df_new["emoji"], df_new["makna"])))
    print(f"✅ {len(unknown_emojis)} emoji baru diterjemahkan dan ditambahkan ke mapping!")
else:
    print("✅ Tidak ada emoji baru ditemukan.")

# === 7. Buat pattern regex untuk semua emoji di mapping ===
emoji_pattern = re.compile("|".join(re.escape(e) for e in emoji_map.keys()))

# === 8. Ganti emoji di dataset ===
def replace_emoji_with_token(text):
    if not isinstance(text, str):
        return text
    text = skin_tone_modifiers.sub("", text)
    return emoji_pattern.sub(lambda m: f" [{emoji_map[m.group()]}] ", text)

df["Komentar"] = df["Komentar"].apply(replace_emoji_with_token)

# === 9. Simpan hasil ===
df.to_csv(output_dataset_file, index=False, encoding="utf-8-sig")
df_map.to_csv(output_mapping_file, index=False, encoding="utf-8-sig")

print(f"✅ Dataset final tersimpan di: {output_dataset_file}")
print(f"✅ Mapping final tersimpan di: {output_mapping_file}")
print(f"✅ Daftar emoji baru di: {output_unknown_file}")


✅ 113 emoji baru diterjemahkan dan ditambahkan ke mapping!
✅ Dataset final tersimpan di: dataset_komentar_emoji_token(1.1).csv
✅ Mapping final tersimpan di: emoji_to_text_token_safe_updated(1.1).csv
✅ Daftar emoji baru di: emoji_unknown(1.1).txt


In [3]:
#Version 2 of the script with Google Translate integration

import pandas as pd
import re
import unicodedata
from googletrans import Translator

# === Konfigurasi file ===
dataset_file = "gabungan_komentar(fix2).csv"  # dataset mentah
mapping_file = "emoji_to_text_token_safe.csv"  # mapping awal
output_dataset_file = "dataset_komentar_emoji_token(2).csv"
output_mapping_file = "emoji_to_text_token_safe_updated.csv"
output_unknown_file = "emoji_unknown.txt"

# === 1. Load mapping awal ===
df_map = pd.read_csv(mapping_file)
emoji_map = dict(zip(df_map["emoji"], df_map["makna"]))

# === 2. Skin tone modifiers ===
skin_tone_modifiers = re.compile(r'[\U0001F3FB-\U0001F3FF]')

# === 3. Translator Google ===
translator = Translator()

# === 4. Deteksi emoji di teks ===
def find_emojis(text):
    return [ch for ch in text if re.match(r'[\U0001F300-\U0001FAFF]', ch)]

# === 5. Proses dataset ===
df = pd.read_csv(dataset_file)
unknown_emojis = set()

for text in df["Komentar"]:
    if isinstance(text, str):
        text_clean = skin_tone_modifiers.sub("", text)
        for e in find_emojis(text_clean):
            if e not in emoji_map:
                unknown_emojis.add(e)

# === 6. Tambahkan emoji baru ke mapping ===
if unknown_emojis:
    new_rows = []
    with open(output_unknown_file, "w", encoding="utf-8") as f:
        for e in sorted(unknown_emojis):
            f.write(f"{e}\n")
            try:
                unicode_name = unicodedata.name(e).replace("_", " ")
            except ValueError:
                unicode_name = "unknown emoji"

            result = translator.translate(unicode_name, src="en", dest="id")
            translated = result.text
            token_name = translated.upper().replace(" ", "_")
            new_rows.append({"emoji": e, "makna": token_name})

    df_new = pd.DataFrame(new_rows)
    df_map = pd.concat([df_map, df_new], ignore_index=True)
    df_map = df_map.drop_duplicates(subset=["emoji"], keep="first")
    emoji_map.update(dict(zip(df_new["emoji"], df_new["makna"])))
    print(f"✅ {len(unknown_emojis)} emoji baru diterjemahkan dan ditambahkan ke mapping!")
else:
    print("✅ Tidak ada emoji baru ditemukan.")

# === 7. Buat pattern regex ===
emoji_pattern = re.compile("|".join(re.escape(e) for e in emoji_map.keys()))

# === 8. Cleaning tambahan: spasi otomatis di sekitar emoji ===
def add_space_around_emoji(text):
    if not isinstance(text, str):
        return text
    text = skin_tone_modifiers.sub("", text)
    text = re.sub(r'([\U0001F300-\U0001FAFF])', r' \1 ', text)  # spasi sebelum & sesudah emoji
    text = re.sub(r'\s+', ' ', text).strip()  # rapikan spasi berlebih
    return text

# === 9. Ganti emoji dengan token ===
def replace_emoji_with_token(text):
    if not isinstance(text, str):
        return text
    text = add_space_around_emoji(text)
    return emoji_pattern.sub(lambda m: f" [{emoji_map[m.group()]}] ", text)

df["Komentar"] = df["Komentar"].apply(replace_emoji_with_token)

# === 10. Simpan hasil ===
df.to_csv(output_dataset_file, index=False, encoding="utf-8-sig")
df_map.to_csv(output_mapping_file, index=False, encoding="utf-8-sig")

print(f"✅ Dataset final tersimpan di: {output_dataset_file}")
print(f"✅ Mapping final tersimpan di: {output_mapping_file}")
print(f"✅ Daftar emoji baru di: {output_unknown_file}")


✅ 113 emoji baru diterjemahkan dan ditambahkan ke mapping!
✅ Dataset final tersimpan di: dataset_komentar_emoji_token(2).csv
✅ Mapping final tersimpan di: emoji_to_text_token_safe_updated.csv
✅ Daftar emoji baru di: emoji_unknown.txt
